# Phase 3: Baseline Model (TF-IDF + Logistic Regression)

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from portfolio_projekt.data import LABELS, THRESHOLD, load_english, load_german
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

## English baseline: word-level TF-IDF
Binary target: toxicity >= 0.5. Trained on the enriched training set, evaluated on
the unmodified validation split.

In [ ]:
df_en_train, df_en_val, df_en_test = load_english()
df_de_train, df_de_val, df_de_test = load_german()

for name, df in [("EN train", df_en_train), ("EN val", df_en_val), ("EN test", df_en_test),
                 ("DE train", df_de_train), ("DE val", df_de_val), ("DE test", df_de_test)]:
    print(f"{name:9} {df.shape}")

In [ ]:
y_en_train = df_en_train["toxicity"] >= THRESHOLD
y_en_val = df_en_val["toxicity"] >= THRESHOLD

vectorizer_word = TfidfVectorizer(min_df=20, sublinear_tf=True)
X_en_train = vectorizer_word.fit_transform(df_en_train["text"])
X_en_val = vectorizer_word.transform(df_en_val["text"])

model_word = LogisticRegression(max_iter=1000)
model_word.fit(X_en_train, y_en_train)
p_en_val = model_word.predict_proba(X_en_val)[:, 1]

print(f"Vocabulary size: {X_en_train.shape[1]}")
print(f"ROC-AUC: {roc_auc_score(y_en_val, p_en_val):.3f}")
print(f"PR-AUC:  {average_precision_score(y_en_val, p_en_val):.3f}")
print(f"F1 at threshold 0.5: {f1_score(y_en_val, p_en_val >= 0.5):.3f}")

In [ ]:
# Precision and recall at the default threshold of 0.5
print(f"At 0.5:  precision {precision_score(y_en_val, p_en_val >= 0.5):.3f}, "
      f"recall {recall_score(y_en_val, p_en_val >= 0.5):.3f}")

# Find the threshold with the best F1 score on the validation data
precision, recall, thresholds = precision_recall_curve(y_en_val, p_en_val)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best = f1[:-1].argmax()

print(f"Best threshold: {thresholds[best]:.2f}")
print(f"  F1 {f1[best]:.3f}, precision {precision[best]:.3f}, recall {recall[best]:.3f}")